# External run, step by step

The steps of `run_cohort` in `src/run_external.py`, one per cell, on one cohort with a short model, so each step can be
inspected or stepped into with "Debug Cell". Nothing is written to `output/`. The last cell runs `run_cohort` itself.

In [ ]:
import argparse
import sys
import tempfile
from pathlib import Path

import pandas as pd
import yaml

import e2m
from e2m.config import load_config

RUN_ROOT = Path.cwd()                      # Runs/External
sys.path.insert(0, str(RUN_ROOT / "src"))
import run_external as R
from cohorts import mutation_labels
from integration import integrate, umap_coordinates

COHORT = "HUGO"                            # any key of config/e2m.yaml cohorts
METHOD = "combat"                          # None = the cohort's main (first) method; ComBat-seq takes ~5-10 min on a CPU
EPOCHS = 2                                 # short model for exploring

## 1. Settings
The run's config, and the model and preprocessing settings from the TCGA run's config, with this run's target rule.

In [ ]:
config = yaml.safe_load((RUN_ROOT / "config" / "e2m.yaml").read_text(encoding="utf-8"))
base = load_config(RUN_ROOT / config["run"]["tcga_config"], {"data": config["data"]})
base["model"]["multitask_nn"]["epochs"] = EPOCHS
settings = config["cohorts"][COHORT]
method = METHOD or settings["methods"][0]
run = config["run"]
settings, method

## 2. The external cohort
The cohort's loader (`src/cohorts.py`): expression, mutation calls, clinical data, metadata and the profiled samples.

In [ ]:
loader, batch_col, site_col = R.LOADERS[COHORT]
expression, mutations, clinical, loader_meta, profiled = loader(RUN_ROOT / run["external_root"] / settings["data_dir"])
batches = clinical[batch_col].astype(str) if batch_col else None
print(expression.shape, mutations.shape, clinical.shape, len(profiled))
{k: v for k, v in loader_meta.items() if k != "measured_genes"}

## 3. TCGA training data
Linear values (no CPM, no log) of the cohort's TCGA cancer types; the CPM and log are applied later, in the integration.

In [ ]:
cancers = base["run"]["cohorts"] if settings["tcga_cancers"] == "all" else settings["tcga_cancers"]
overrides = {**config["data"], "expression_dataset": settings["tcga_expression"], "expression_transform": "raw", "normalization": "none"}
tcga = e2m.Dataset.from_tcga(cancers, data_dir=RUN_ROOT / run["tcga_data_dir"], config=RUN_ROOT / run["tcga_config"],
                             data_overrides=overrides, with_tmb=False)
targets = tcga.targets.tolist()
tcga, len(targets)

## 4. External labels
1/0 per target for profiled samples, missing for the others (and, for a gene panel, for genes outside it).

In [ ]:
labels = mutation_labels(mutations, profiled, expression.index, targets)
if "measured_genes" in loader_meta:
    labels.loc[:, ~labels.columns.isin(loader_meta["measured_genes"])] = float("nan")
ranking = settings.get("target_ranking")
top = R.top_targets(RUN_ROOT / ranking["file"], ranking["top_n"]) if ranking and (RUN_ROOT / ranking["file"]).exists() else None
print("profiled samples:", int(labels.notna().any(axis=1).sum()), "of", len(labels))
labels.sum().sort_values(ascending=False).head(10)

## 5. Integration (`src/integration.py`)
TCGA and the cohort on the shared genes, brought to the same scale with the chosen method.

In [ ]:
scales = {"tcga_scale": "counts" if settings["tcga_expression"] == "star_counts" else "tpm",
          "external_scale": settings["scale"], "cpm": base["data"].get("normalization") == "cpm"}
train, test = integrate(tcga.expression, expression, method, external_batches=batches, **scales)
train.shape, test.shape

## 6. Model
An E2M model trained on all TCGA samples, applied to the cohort.

In [ ]:
model = e2m.E2MModel(base).fit(train, tcga.mutations.loc[train.index])
probabilities = model.predict(test)
pd.DataFrame(model.history).tail()

## 7. Metrics
Per target, on the profiled samples; per site for CPTAC/CMI and ImmunoPOG.

In [ ]:
metrics = R.evaluate(labels.loc[test.index], probabilities, targets, top)
if site_col:
    sites = R.site_metrics(labels.loc[test.index], probabilities, targets, clinical[site_col].reindex(test.index), run["min_site_samples"])
print(R.summarize(metrics))
metrics[metrics["evaluable"]].sort_values("normalized_auprc", ascending=False).head(10)

## 8. Embeddings and UMAP
The model's last hidden layer for TCGA and the cohort, and UMAP coordinates before and after integration (about a minute each).

In [ ]:
embeddings_external = model.embed(test)
before = umap_coordinates(*integrate(tcga.expression, expression, "none", **scales), batches, label=COHORT)
after = umap_coordinates(train, test, batches, label=COHORT)
embeddings_external.shape, before.head()

## 9. The whole step
`run_cohort` as the run calls it, writing to a temporary folder.

In [ ]:
# The melanoma target ranking needs the TCGA-SKCM run; without it, the step runs without the ranking.
run_settings = {k: v for k, v in settings.items() if k != "target_ranking" or (RUN_ROOT / v["file"]).exists()}
args = argparse.Namespace(output_dir=Path(tempfile.mkdtemp()), methods=[method], skip_umap=True)
rows = R.run_cohort(COHORT, run_settings, config, base, args, [method])
print(args.output_dir)
pd.DataFrame(rows)